In [2]:
import os
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from shapely.ops import linemerge, unary_union


In [ ]:
input_file = "Foundation_Data.gpkg"
coastline_edge_layer = gpd.read_file(input_file, layer="edges")  

# Merge all line strings into one
merged_line = linemerge(unary_union(coastline_edge_layer.geometry))

# Convert to GeoDataFrame
merged_gdf = gpd.GeoDataFrame(geometry=[merged_line], crs=coastline_edge_layer.crs)

# Save or process further
merged_gdf.to_file("data/Jamaica_Coastline_Layer.gpkg", driver="GPKG")


In [10]:
import geopandas as gpd
from shapely.geometry import Point
import pandas as pd

# Load the GeoPackage layer
input_gpkg = "/Users/adam/Desktop/coastlines/Coastal_layer_segments.gpkg"  # Replace with your file path
edge_layer_name = "0.25km"    # Replace with your edge layer name
edges = gpd.read_file(input_gpkg, layer=edge_layer_name)

# Create a DataFrame to store unique nodes
node_set = {}

# Function to round coordinates to a fixed number of decimals
def round_coords(coords, decimals=5):
    return tuple(round(c, decimals) for c in coords)

# Function to get or create a node ID for a point
def get_node_id(point, node_counter, decimals=5):
    rounded_point = round_coords(point, decimals)
    if rounded_point not in node_set:
        node_id = f"node_{node_counter[0]}"
        node_set[rounded_point] = node_id
        node_counter[0] += 1
        return node_id
    return node_set[rounded_point]

# Initialize node counter
node_counter = [1]

# Add from_node and to_node fields to edges
edges["from_id"] = ""
edges["to_id"] = ""
edges["id"] = ""

for index, row in edges.iterrows():
    line = row.geometry
    if line is not None and line.geom_type == "LineString":
        # Extract the start and end points
        start_point = tuple(line.coords[0])
        end_point = tuple(line.coords[-1])

        # Assign node IDs
        start_node_id = get_node_id(start_point, node_counter)
        end_node_id = get_node_id(end_point, node_counter)


        # Add IDs to the edge
        edges.at[index, "from_id"] = start_node_id
        edges.at[index, "to_id"] = end_node_id
        edges.at[index, "id"] = "edge_" + str(index)

# Create node GeoDataFrame
nodes = gpd.GeoDataFrame(
    [{"node_id": node_id, "geometry": Point(coords)} for coords, node_id in node_set.items()],
    crs=edges.crs
)

# Save the updated layers to a new GeoPackage
output_gpkg = "data/Foundation_Data.gpkg"
edges.to_file(output_gpkg, layer="edges", driver="GPKG")
nodes.to_file(output_gpkg, layer="nodes", driver="GPKG")

print("Node and edge layers have been successfully created and saved.")


Node and edge layers have been successfully created and saved.
